# Calculating Network Error with Loss

With a randomly-initialized model, or even a model initialized with more sophisticated 
approaches, our goal is to train, or teach, a model over time. To train a model, we tweak the 
weights and biases to improve the model’s accuracy and confidence. To do this, we calculate how 
much error the model has. The ​ loss function
, also referred to as the ​ cost function
algorithm that quantifies how wrong a model is. ​ Loss
, is the 
 is the measure of this metric. Since loss is 
the model’s error, we ideally want it to be 0.

# Categorical Cross-Entropy Loss

If you’re familiar with linear regression, then you already know one of the loss functions used 
with neural networks that do regression: ​ squared error
 (or ​ mean squared error
networks). 

<br>

We’re not performing regression in this example; we’re classifying, so we need a different loss 
function. The model has a softmax activation function for the output layer, which means it’s 
outputting a probability distribution. ​ Categorical cross-entropy
 is explicitly used to compare 
a “ground-truth” probability (​y ​ or​ ​ “​targets​”) and some predicted distribution (​y-hat ​ or 
“​predictions​”), so it makes sense to use cross-entropy here. It is also one of the most 
commonly used loss functions with a softmax activation on the output layer. 

<br>

The formula for calculating the categorical cross-entropy of ​ y​ (actual/desired distribution) and 
y-hat​ (predicted distribution) is:

$$
L(y, \hat{y}) = -\sum_{i} y_i \log(\hat{y}_i)
$$

Where:
- $y$ is the true probability distribution (one-hot encoded vector)
- $\hat{y}$ is the predicted probability distribution (output of softmax)
- $i$ iterates over each class

### For a batch of samples:

$$
L = -\frac{1}{N} \sum_{j=1}^{N} \sum_{i} y_{i}^{(j)} \log(\hat{y}_{i}^{(j)})
$$

Where $N$ is the number of samples.

### Simplified (since $y$ is one-hot encoded):

When $y$ is one-hot encoded (only one class is correct), this simplifies to:

$$
L = -\log(\hat{y}_c)
$$

Where $\hat{y}_c$ is the predicted probability of the correct class.

<br>

You may ask why we call this cross-entropy and not ​ log loss
, which is also a type of loss. If you 
do not know what log loss is, you may wonder why there is such a fancy looking formula for 
what looks to be a fairly basic description.

<br>

In general, the log loss error function is what we apply to the output of a binary logistic regression 
model (which we’ll describe in chapter 16)  — there are only two classes in the distribution, each 
of them applying to a single output (neuron) which is targeted as a 0 or 1. In our case, we have a 
classification model that returns a probability distribution over all of the outputs. Cross-entropy 
compares two probability distributions. In our case, we have a softmax output, let’s say it’s: 

``` bash
softmax_output = [0.7, 0.1, 0.2]
```
<br>

Let’s see the Python code for this:


In [5]:
import math

# an example output from the output layer of the neural network
softmax_output = [0.7, 0.1, 0.2]
# ground truth
target_output = [1, 0, 0]

loss = -(math.log(softmax_output[0]) * target_output[0] +
         math.log(softmax_output[1]) * target_output[1] +
         math.log(softmax_output[2]) * target_output[2])

print(loss)

0.35667494393873245


So far, we’ve applied log() to the softmax output, but have neither explained what “log” is nor 
why we use it. We will save the discussion of “why” until the next chapter, which covers 
derivatives, gradients, and optimizations; suffice it to say that the log function has some desirable 
properties. ​ Log
 is short for ​ logarithm 
and is defined as the solution for the x-term in an equation 
of the form a​x ​= b. For example, ​ 10​x ​= 100​ can be solved with a log: ​ log​10​
(100)​, which evaluates to 
2. 

In [9]:
import numpy as np

b = 5.2
print(np.log(b))

# We can confirm this by exponentiating our result:

import math
print(math.exp(1.6486586255873816))

1.6486586255873816
5.2


In [ ]:
softmax_outputs = [[0.7, 0.1, 0.2],
                   [0.1, 0.5, 0.4],
                   [0.02, 0.9, 0.08]]

class_targets = [0,0,1]

for targ_idx, distribution in zip(class_targets, softmax_outputs):
    loss = -math.log(distribution[targ_idx])
    print(loss)


0.35667494393873245
2.3025850929940455
0.10536051565782628


The ​zip​()​ function, again, lets us iterate over multiple iterables at the same time in Python. This 
can be further simplified using NumPy (we’re creating a NumPy array of the Softmax outputs this 
time): 

In [5]:
import numpy as np
softmax_outputs = np.array([[0.7, 0.1, 0.2],
                   [0.1, 0.5, 0.4], 
                   [0.02, 0.9, 0.08]])
class_targets = [0, 1, 2]


print(softmax_outputs[[0,1,2], class_targets])

[0.7  0.5  0.08]


In [6]:
print(softmax_outputs[
    range(len(softmax_outputs)),
    class_targets
])

[0.7  0.5  0.08]


In [7]:
print(-np.log(softmax_outputs[
    range(len(softmax_outputs)),
    class_targets
]))

[0.35667494 0.69314718 2.52572864]


Finally, we want an average loss per batch to have an idea about how our model is doing during 
training. There are many ways to calculate an average in Python; the most basic form of an 
average is the ​ arithmetic mean
: ​ sum(iterable) / len(iterable)​. NumPy has a method that 
computes this average on arrays, so we will use that instead. We add NumPy’s average to the 
code:

In [8]:
neg_log = -np.log(softmax_outputs[
    range(len(softmax_outputs)),
    class_targets
])

avg_loss = np.mean(neg_log)

print(avg_loss)

1.191850256268978


In [11]:
import numpy as np

softmax_outputs = np.array([[0.7,0.1,0.2],
                            [0.1,0.5,0.4],
                            [0.02,0.9,0.08]])

class_targets = np.array([[1,0,0],
                         [0,1,0],
                         [0,1,0]])

# Probabilities for target values - 
# only if categorical labels
if len(class_targets.shape) == 1:
    correct_confidences = softmax_outputs[
        range(len(softmax_outputs)),
        class_targets
    ]

# Mask values - only for one-hot encoded labels
elif len(class_targets.shape) == 2:
    correct_confidences = np.sum(softmax_outputs*class_targets, axis=1)

neg_log = -np.log(correct_confidences)

avg_loss = np.mean(neg_log)



In [12]:
import numpy as np

-np.log(0)

C:\Users\AGP KOHAT\AppData\Local\Temp\ipykernel_10680\3197048679.py:3: RuntimeWarning: divide by zero encountered in log
  -np.log(0)


np.float64(inf)

In [13]:
np.e**(-np.inf)

0.0

In programming, the fewer things that are undefined, the better. Later on, we’ll see similar 
simplifications, for example when calculating a derivative of the absolute value function, which 
does not exist for an input of ​ 0​ and we’ll have to make some decisions to work around this. 

# The Categorical Cross-Entropy Loss Class 

 no matter which loss function we’ll use, the overall loss is always a mean value of all 
sample losses. Let’s create the ​Loss​ class containing the ​calculate​ method that will call our 
loss object’s forward method and calculate the mean value of the returned sample losses: 

In [14]:
# common loss class:
class Loss:
    # calculates the data and regularization losses
    # given model output and ground truth values
    def calculte(self, output, y):
        # calculate sample losses
        sample_losses = self.forward(output, y)

        # calculate mean loss
        data_loss = np.mean(sample_losses)

        # return loss
        return data_loss

Let’s convert our loss code into a class for convenience down the line

In [15]:
# cross-entropy loss
class Loss_CategoricalCrossentropy(Loss):
    # forward pass
    def forward(self, y_pred, y_true):
        
        # number of samples in a batch
        samples = len(y_pred)

        # clip data to prevent division by 0
        # clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        # probabilities for target values -
        # only if categorical labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples),
                y_true
            ]
        # mask labels for one hot encoded labels
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)
        
        # losses
        neg_log = -np.log(correct_confidences)
        return neg_log

In [17]:
loss_func = Loss_CategoricalCrossentropy()
loss = loss_func.forward(softmax_outputs, class_targets)
print(loss)

[0.35667494 0.69314718 0.10536052]


# Combining everything up to this point:

In [18]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

# Dense layer
class Layer_Dense:
    # Layer initialization
    def __init__(self, n_inputs, n_neurons):
        # Initialize weights and biases
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    # Forward pass
    def forward(self, inputs):
        # Calculate output values from inputs, weights and biases
        self.output = np.dot(inputs, self.weights) + self.biases

# ReLU activation
class Activation_ReLU:
    # Forward pass
    def forward(self, inputs):
        # Calculate output values from inputs
        self.output = np.maximum(0, inputs)

# Softmax activation
class Activation_Softmax:
    # Forward pass
    def forward(self, inputs):
        # Get unnormalized probabilities
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        # Normalize them for each sample
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probabilities

# Common loss class
class Loss:
    # Calculates the data and regularization losses
    # given model output and ground truth values
    def calculate(self, output, y):
        # Calculate sample losses
        sample_losses = self.forward(output, y)
        # Calculate mean loss
        data_loss = np.mean(sample_losses)
        # Return loss
        return data_loss

# Cross-entropy loss
class Loss_CategoricalCrossentropy(Loss):
    # Forward pass
    def forward(self, y_pred, y_true):
        # Number of samples in a batch
        samples = len(y_pred)
        # Clip data to prevent division by 0
        # Clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        # Probabilities for target values -
        # only if categorical labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples), y_true
            ]
        # Mask values - only for one-hot encoded labels
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true, axis=1
            )

        # Losses
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods

# Create dataset
X, y = spiral_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense1 = Layer_Dense(2, 3)

# Create ReLU activation (to be used with Dense layer):
activation1 = Activation_ReLU()

# Create second Dense layer with 3 input features (as we take output
# of previous layer here) and 3 output values
dense2 = Layer_Dense(3, 3)

# Create Softmax activation (to be used with Dense layer):
activation2 = Activation_Softmax()

# Create loss function
loss_function = Loss_CategoricalCrossentropy()

# Perform a forward pass of our training data through this layer
dense1.forward(X)

# Perform a forward pass through activation function
# it takes the output of first dense layer here
activation1.forward(dense1.output)

# Perform a forward pass through second Dense layer
# it takes outputs of activation function of first layer as inputs
dense2.forward(activation1.output)

# Perform a forward pass through activation function
# it takes the output of second dense layer here
activation2.forward(dense2.output)

# Let's see output of the first few samples:
print(activation2.output[:5])

# Perform a forward pass through loss function
# it takes the output of second dense layer here and returns loss
loss = loss_function.calculate(activation2.output, y)

# Print loss value
print('loss:', loss)

[[0.33333334 0.33333334 0.33333334]
 [0.3333332  0.3333332  0.33333364]
 [0.3333329  0.33333293 0.3333342 ]
 [0.3333326  0.33333263 0.33333477]
 [0.33333233 0.3333324  0.33333528]]
loss: 1.0986104


# Accuracy Calculation

While loss is a useful metric for optimizing a model, the metric commonly used in practice along 
with loss is the ​accuracy​, which describes how often the largest confidence is the correct class 
in terms of a fraction. Conveniently, we can reuse existing variable definitions to calculate the 
accuracy metric. We will use the ​argmax ​values from the ​softmax outputs ​and then compare 
these to the targets. This is as simple as doing (note that we slightly modified the 
softmax_outputs​ for the purpose of this example):

In [20]:
import numpy as np

# probabilities for 3 samples 
softmax_output = np.array([[0.7, 0.1, 0.2],
                           [0.1, 0.5, 0.4],
                            [0.02, 0.9, 0.08]])

# Target labels for 3 samples
class_targets = np.array([[1, 0, 0]])

# calculate values along second axis (axis of index 1)

predictions = np.argmax(softmax_output, axis=1)

#  if targets are one-hot encoded, turn them into discrete values
if len(class_targets.shape) == 2:
    class_targets = np.argmax(class_targets, axis=1)
# True evaluates to1; False to 0
accuracy = np.mean(predictions==class_targets)

print('acc', accuracy)

acc 0.3333333333333333


we can add the following to the ed of our full script above to calculate it's accuracy 


In [21]:
# Calculate accuracy from output of activation2 and targets
# Calculate values along first axis
predictions = np.argmax(activation2.output, axis=1)
if len(y.shape) == 2:
    y = np.argmax(y, axis=1)
accuracy = np.mean(predictions == y)
# Print accuracy
print('acc:', accuracy)

acc: 0.34
